# Legal Vakta RAG Experiments

Use this notebook to inspect PDF loading, chunking, embeddings, and retrieval quality.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import Settings
from src.ingestion.pdf_loader import find_pdf_files, DocumentProcessor
from src.retrieval.vectorstore import create_vectorstore, get_retriever
from src.retrieval.retriever import LegalRetriever

Settings.pdf_dir

## Load Sample PDFs

In [ ]:
pdfs = find_pdf_files(Settings.pdf_dir)
print(f'Total PDFs: {len(pdfs)}')
pdfs[:5]

## Chunking Debug

In [ ]:
processor = DocumentProcessor(chunk_size=900, chunk_overlap=150)
docs = []
for pdf in pdfs[:3]:
    docs.extend(processor.load_documents(pdf.parent))
    break

chunks = processor.split_documents(docs[:10])
print(f'Pages loaded: {len(docs)}')
print(f'Chunks created: {len(chunks)}')
print(chunks[0].metadata)
print(chunks[0].page_content[:1000])

## Embedding and Retrieval Smoke Test

In [ ]:
sample_chunks = chunks[:50]
vectorstore = create_vectorstore(sample_chunks, PROJECT_ROOT / 'vectorstore' / 'notebook_test')
retriever = LegalRetriever(get_retriever(vectorstore, k=3))

results = retriever.retrieve('benefit of doubt in criminal appeal')
for result in results:
    print(result['metadata'])
    print(result['content'][:500])
    print('-' * 80)